In [ ]:
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [ ]:
!nvidia-smi
!git clone https://github.com/Saad1926Q/llm-emotion-representations.git
%cd /kaggle/working/llm-emotion-representations

In [ ]:
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTHONPATH"] = (
    "/kaggle/working/llm-emotion-representations/src"
)

In [ ]:
import torch
import transformers

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
!python scripts/generate_dataset.py \
  --run-dir /kaggle/working/qwen35_2b_fp32 \
  --torch-dtype float32 \
  --device-map cuda \
  --batch-size 2 \
  --limit 4

In [ ]:
!python scripts/generate_dataset.py \
  --run-dir /kaggle/working/qwen35_2b_fp32 \
  --torch-dtype float32 \
  --device-map cuda \
  --batch-size 8

In [ ]:
!wc -l /kaggle/working/qwen35_2b_fp32/generations.jsonl

In [ ]:
!python scripts/extract_activations.py \
  --run-dir /kaggle/working/qwen35_2b_fp32 \
  --torch-dtype float32 \
  --device-map cuda \
  --batch-size 2 \
  --limit 4

In [ ]:
!python scripts/extract_activations.py \
  --run-dir /kaggle/working/qwen35_2b_fp32 \
  --torch-dtype float32 \
  --device-map cuda \
  --batch-size 4

In [ ]:
from pathlib import Path

activation_dir = Path(
    "/kaggle/working/qwen35_2b_fp32/activations"
)
print("Activation files:", len(list(activation_dir.glob("*.pt"))))

In [ ]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import HfApi, login



repo_id = "saad1926q/qwen35-2b-emotion-vectors"

HfApi().create_repo(
    repo_id=repo_id,
    repo_type="model",
    exist_ok=True,
)

In [ ]:
!python scripts/build_vectors.py \
  --run-dir /kaggle/working/qwen35_2b_fp32 \
  --hf-repo saad1926q/qwen35-2b-emotion-vectors

In [ ]:
from huggingface_hub import snapshot_download

snapshot_download(
    repo_id="saad1926q/qwen35-2b-emotion-vectors",
    local_dir="/kaggle/working/frozen_vectors",
)

In [ ]:
!python scripts/generate_dataset_b.py \
  --run-dir /kaggle/working/dataset_b \
  --torch-dtype float32 \
  --device-map cuda \
  --batch-size 2 \
  --limit-conversations 2

In [ ]:
!python scripts/generate_dataset_b.py \
  --run-dir /kaggle/working/dataset_b \
  --torch-dtype float32 \
  --device-map cuda \
  --batch-size 4

In [ ]:
!python scripts/score_internal_b.py \
  --run-dir /kaggle/working/dataset_b \
  --vector-dir /kaggle/working/frozen_vectors \
  --torch-dtype float32 \
  --device-map cuda \
  --batch-size 2 \
  --limit 20

In [ ]:
!wc -l /kaggle/working/dataset_b/internal_scores.jsonl

In [ ]:
!python scripts/score_internal_b.py \
  --run-dir /kaggle/working/dataset_b \
  --vector-dir /kaggle/working/frozen_vectors \
  --torch-dtype float32 \
  --device-map cuda \
  --batch-size 2

In [ ]:
!wc -l /kaggle/working/dataset_b/internal_scores.jsonl

In [ ]:
from pathlib import Path

def count_lines(path):
    with open(path, encoding="utf-8") as f:
        return sum(1 for line in f if line.strip())

print(
    "Dataset A generations:",
    count_lines("/kaggle/working/qwen35_2b_fp32/generations.jsonl"),
)
print(
    "Dataset A activations:",
    len(list(Path(
        "/kaggle/working/qwen35_2b_fp32/activations"
    ).glob("*.pt"))),
)
print(
    "Dataset B manifest:",
    count_lines("/kaggle/working/dataset_b/manifest.jsonl"),
)
print(
    "Dataset B generations:",
    count_lines("/kaggle/working/dataset_b/generations.jsonl"),
)
print(
    "Internal scores:",
    count_lines("/kaggle/working/dataset_b/internal_scores.jsonl"),
)


In [ ]:
!tar -czf /kaggle/working/ppt2_essential.tar.gz \
  -C /kaggle/working/qwen35_2b_fp32 vectors \
  -C /kaggle/working dataset_b/manifest.jsonl \
  dataset_b/generations.jsonl \
  dataset_b/internal_scores.jsonl


In [ ]:
!tar -czf /kaggle/working/dataset_a_reproducibility.tar.gz \
  -C /kaggle/working qwen35_2b_fp32